# OBPOTF Simulations on Bivariate Bicycle Codes

This notebook runs BP+BP+OTF Monte Carlo simulations on the bivariate bicycle
codes from [Bravyi et al.](https://arxiv.org/abs/2308.07915):

| Code | Parameters | Distance |
|------|-----------|----------|
| BB-72 | [[72, 12, 6]] | 6 |
| BB-108 | [[108, 8, 10]] | 10 |
| BB-144 | [[144, 12, 12]] | 12 |
| BB-288 | [[288, 12, 18]] | 18 |

Pre-computed transfer matrices are loaded from `transfer_matrices/`.
Circuit construction uses functions from the
[SlidingWindowDecoder](https://github.com/gongaa/SlidingWindowDecoder) package.

Reference: deMarti iOlius et al., [arXiv:2409.01440](https://arxiv.org/abs/2409.01440)

## 0. Install SlidingWindowDecoder (if needed)

```bash
pip install git+https://github.com/gongaa/SlidingWindowDecoder.git
```

In [ ]:
import numpy as np
import scipy.io as sio
from timeit import default_timer as timer

from SlidingWindowDecoder.src.codes_q import create_bivariate_bicycle_codes
from SlidingWindowDecoder.src.build_circuit import build_circuit

from BPOTF import OBPOTF, NoiseType, DemData
from BPOTF import __version__ as bpotf_version
from ldpc.ckt_noise.dem_matrices import detector_error_model_to_check_matrices

print(f"BPOTF version: v{bpotf_version}")

## 1. Define the BB code configurations

In [ ]:
BB_CODES = {
    72: {
        "l": 6, "m": 6,
        "A_x_pows": [3], "A_y_pows": [1, 2],
        "B_x_pows": [1, 2], "B_y_pows": [3],
        "d": 6,
        "transfer_file": "transfer_matrices/PhenoTransf72Test.mat",
    },
    108: {
        "l": 9, "m": 6,
        "A_x_pows": [3], "A_y_pows": [1, 2],
        "B_x_pows": [1, 2], "B_y_pows": [3],
        "d": 10,
        "transfer_file": "transfer_matrices/PhenoTransf108Test.mat",
    },
    144: {
        "l": 12, "m": 6,
        "A_x_pows": [3], "A_y_pows": [1, 2],
        "B_x_pows": [1, 2], "B_y_pows": [3],
        "d": 12,
        "transfer_file": "transfer_matrices/PhenoTransf144Test.mat",
    },
    288: {
        "l": 12, "m": 12,
        "A_x_pows": [3], "A_y_pows": [2, 7],
        "B_x_pows": [1, 2], "B_y_pows": [3],
        "d": 18,
        "transfer_file": "transfer_matrices/PhenoTransf288Test.mat",
    },
}

## 2. Helper: build decoder for a BB code

In [ ]:
def build_bb_decoder(bb_type, p, bp_iterations=None, decimation=1e-9):
    """Build a circuit, DEM, and OBPOTF decoder for a given BB code.

    Parameters
    ----------
    bb_type : int
        Code block length (72, 108, 144, or 288).
    p : float
        Physical error rate.
    bp_iterations : list of 3 ints or None
        Max BP iterations for [DEM stage, phenomenological stage, OTF stage].
    decimation : float
        Decimation parameter for the OTF stage. Controls the initial LLR
        value assigned to columns that are NOT selected by the OTF
        (Kruskal) algorithm. A value of 0 means unselected columns are
        completely suppressed; larger values allow a small residual
        probability for unselected columns during the final BP round.

    Returns
    -------
    decoder : OBPOTF
    circuit : stim.Circuit
    bm : check matrix bundle
    """
    cfg = BB_CODES[bb_type]

    # Build code and circuit using SlidingWindowDecoder
    code, A_list, B_list = create_bivariate_bicycle_codes(
        cfg["l"], cfg["m"],
        cfg["A_x_pows"], cfg["A_y_pows"],
        cfg["B_x_pows"], cfg["B_y_pows"],
    )
    d = cfg["d"]
    circuit = build_circuit(code, A_list, B_list, p=p, num_repeat=d, z_basis=True)
    dem = circuit.detector_error_model()
    bm = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)

    # Load pre-computed transfer matrix and phenomenological matrices
    mat_data = sio.loadmat(cfg["transfer_file"])
    transfer_mat = mat_data["transfMatDEMtoPheno"]
    if hasattr(transfer_mat, 'toarray'):
        transfer_mat = transfer_mat.toarray('F')
    phen_check = mat_data["dem_pheno"]
    phen_obs = mat_data["obsphen"]

    # Build DemData
    dem_data = DemData()
    dem_data.priors = bm.priors
    dem_data.obs_matrix = bm.observables_matrix.toarray('F').astype(np.uint8)
    dem_data.transfer_matrix = transfer_mat.astype(np.uint8)
    dem_data.phen_check_matrix = phen_check.astype(np.uint8)
    dem_data.phen_obs_matrix = phen_obs.astype(np.uint8)

    # BP iteration counts
    if bp_iterations is not None:
        bp_iters = np.array(bp_iterations, dtype=np.int32)
    else:
        bp_iters = None

    decoder = OBPOTF(
        bm.check_matrix,
        p,
        NoiseType.E_CLN,
        ps_ext_dem_data=dem_data,
        po_ext_bp_iters=bp_iters,
        decimation=decimation,
    )

    return decoder, circuit, bm

## 3. Run simulations

In [ ]:
# Simulation parameters
bb_types = [72, 108, 144, 288]
p_values = [1e-3, 2e-3, 3e-3]
NMC = 5000  # Monte Carlo shots per configuration
bp_iterations = [100, 400, 100]  # [DEM stage, pheno stage, OTF stage]

# Decimation: controls the initial LLR assigned to columns not selected by OTF.
# A value of 1e-9 assigns a near-zero probability to unselected columns,
# effectively suppressing them while keeping numerical stability.
decimation = 1e-9

results = {}

for bb_type in bb_types:
    d = BB_CODES[bb_type]["d"]
    results[bb_type] = {}

    for p in p_values:
        print(f"\n{'='*60}")
        print(f"BB-{bb_type} [[{bb_type}, *, {d}]] @ p = {p}")
        print(f"{'='*60}")

        decoder, circuit, bm = build_bb_decoder(bb_type, p, bp_iterations, decimation)
        sampler = circuit.compile_detector_sampler()

        num_errors = 0
        total_time = 0.0

        detection_events, observable_flips = sampler.sample(
            NMC, separate_observables=True
        )

        start = timer()
        for i in range(NMC):
            prediction = decoder.decode(detection_events[i].astype(np.uint8))
            if not np.all(prediction == observable_flips[i]):
                num_errors += 1
        total_time = timer() - start

        error_rate = num_errors / NMC
        error_rate_per_round = error_rate / d
        avg_time_per_shot = total_time / NMC

        results[bb_type][p] = {
            "shots": NMC,
            "errors": num_errors,
            "error_rate": error_rate,
            "error_rate_per_round": error_rate_per_round,
            "total_time": total_time,
            "avg_time_per_shot": avg_time_per_shot,
        }

        print(f"  Shots: {NMC}, Errors: {num_errors}")
        print(f"  p(e): {error_rate:.6f}")
        print(f"  p(e)/d: {error_rate_per_round:.6f}")
        print(f"  Total time: {total_time:.2f}s, Avg: {avg_time_per_shot*1000:.2f} ms/shot")

## 4. Results table

In [ ]:
print(f"{'Code':<10} {'d':<5} {'p':<10} {'shots':<8} {'errors':<8} {'p(e)':<12} {'p(e)/d':<12} {'ms/shot':<10}")
print("-" * 80)

for bb_type in bb_types:
    d = BB_CODES[bb_type]["d"]
    for p in p_values:
        r = results[bb_type][p]
        print(
            f"BB-{bb_type:<6} {d:<5} {p:<10.4f} {r['shots']:<8} {r['errors']:<8} "
            f"{r['error_rate']:<12.6f} {r['error_rate_per_round']:<12.8f} "
            f"{r['avg_time_per_shot']*1000:<10.2f}"
        )

## 5. Plot logical error rate per round

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))

for bb_type in bb_types:
    d = BB_CODES[bb_type]["d"]
    ps = sorted(results[bb_type].keys())
    rates = [results[bb_type][p]["error_rate_per_round"] for p in ps]
    # Only plot points with at least 1 error
    valid = [(pp, rr) for pp, rr in zip(ps, rates) if rr > 0]
    if valid:
        vp, vr = zip(*valid)
        ax.plot(vp, vr, "o-", label=f"[[{bb_type}, *, {d}]]")

ax.set_xlabel("Physical error rate")
ax.set_ylabel("Logical error rate per round")
ax.set_xscale("log")
ax.set_yscale("log")
ax.legend()
ax.set_title("BP+BP+OTF -- Bivariate Bicycle Codes")
ax.grid(True, which="both", ls="--", alpha=0.5)
plt.tight_layout()
plt.show()